# 09 — Mention and Reply Networks

**Project:** From Online Attention to Electoral Outcomes: A Network Science Analysis of Philippine Election 2025 Twitter/X Communication

This notebook is part of the notebook set for the Philippine Election 2025 Twitter/X network science project.



## Visible progress tracker

This notebook now prints numbered stage updates while it runs. If Jupyter shows `[*]`, the current stage is still processing. A run log is also saved under `outputs/run_logs/`.


In [ ]:
# VISIBLE PROGRESS TRACKER — automatically added in v5
from pathlib import Path as _ProgressPath
import sys as _progress_sys
_PROGRESS_ROOT = _ProgressPath.cwd().parent if _ProgressPath.cwd().name == "notebooks" else _ProgressPath.cwd()
_progress_sys.path.append(str(_PROGRESS_ROOT / "src"))
try:
    from election_network_progress import make_tracker
    progress = make_tracker("09_mention_reply_network", total_steps=8, root=_PROGRESS_ROOT)
except Exception as _progress_error:
    print(f"Progress tracker could not start: {_progress_error}")
    class _FallbackProgress:
        def __init__(self): self.current = 0
        def step(self, label):
            self.current += 1
            print(f"[stage {self.current}] {label}", flush=True)
        def info(self, label): print(f"[info] {label}", flush=True)
        def done(self, label="Notebook completed"): print(f"✓ {label}", flush=True)
    progress = _FallbackProgress()


## Purpose

This notebook builds user-level communication networks: mention network and reply network. It uses obfuscated author IDs for privacy-preserving analysis and overlays known public authors where available.

In [ ]:
progress.step("Purpose")
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")

# Make ../src importable when running from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT / "src"))

from election_network_utils import *
paths = ensure_dirs(ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)
px.defaults.template = "plotly_white"

RAW = paths["raw"]
PROCESSED = paths["processed"]
FIGURES = paths["figures"]
INTERACTIVE = paths["interactive"]
TABLES = paths["tables"]
NETWORKS = paths["networks"]
REPORT_ASSETS = paths["report_assets"]

print("Project root:", ROOT)


In [ ]:
progress.step("Purpose")
tweets = pd.read_csv(PROCESSED / "tweets_with_entities.csv", parse_dates=["createdAt"])
tweets = parse_list_columns(tweets, ["mentions"])
known = pd.read_csv(PROCESSED / "known_authors_activity_summary.csv") if (PROCESSED / "known_authors_activity_summary.csv").exists() else pd.DataFrame()
print(tweets.shape)


In [ ]:
progress.step("Purpose")
# Mention network: pseudo author -> mentioned handle extracted from text.
mention_counter = Counter()
for _, row in tweets.iterrows():
    author = str(row.get("pseudo_author_userName", "")).replace("@", "")
    if not author or author == "nan":
        continue
    for m in row["mentions"]:
        mention_counter[(author, "@" + str(m).lower())] += 1

mention_edges = pd.DataFrame([{"source": s, "target": t, "weight": w} for (s, t), w in mention_counter.items()])
mention_edges.to_csv(PROCESSED / "user_mention_edges_full.csv", index=False)
mention_edges_filtered = mention_edges[mention_edges["weight"] >= 2].copy() if not mention_edges.empty else mention_edges
mention_edges_filtered.to_csv(PROCESSED / "user_mention_edges_filtered_weight2.csv", index=False)
print("Mention edges:", len(mention_edges), "Filtered:", len(mention_edges_filtered))
mention_edges_filtered.head(20)


In [ ]:
progress.step("Purpose")
MG = graph_from_edges(mention_edges_filtered, source="source", target="target", weight="weight", directed=True)
mention_summary = network_summary(MG, "user_mention_network")
mention_metrics = compute_network_metrics(MG)
mention_metrics.to_csv(PROCESSED / "user_mention_node_metrics.csv", index=False)
mention_summary.to_csv(TABLES / "09_user_mention_network_summary.csv", index=False)
nx.write_gexf(MG, NETWORKS / "user_mention_network.gexf")
display(mention_summary)
display(mention_metrics.head(20))


In [ ]:
progress.step("Purpose")
# Reply network: pseudo author -> pseudo user being replied to
reply_df = tweets.dropna(subset=["pseudo_inReplyToUsername"]).copy()
reply_df["source"] = reply_df["pseudo_author_userName"].astype(str).str.replace("@", "", regex=False)
reply_df["target"] = reply_df["pseudo_inReplyToUsername"].astype(str).str.replace("@", "", regex=False)
reply_edges = reply_df.groupby(["source", "target"], as_index=False).size().rename(columns={"size": "weight"})
reply_edges = reply_edges[(reply_edges["source"] != "nan") & (reply_edges["target"] != "nan")]
reply_edges.to_csv(PROCESSED / "reply_edges_full.csv", index=False)
reply_edges_filtered = reply_edges[reply_edges["weight"] >= 2].copy()
reply_edges_filtered.to_csv(PROCESSED / "reply_edges_filtered_weight2.csv", index=False)
print("Reply edges:", len(reply_edges), "Filtered:", len(reply_edges_filtered))
reply_edges_filtered.head(20)


In [ ]:
progress.step("RG = graph_from_edges(reply_edges_filtered, source='source', target='target', weight='weight', directed=True)")
RG = graph_from_edges(reply_edges_filtered, source="source", target="target", weight="weight", directed=True)
reply_summary = network_summary(RG, "reply_network")
reply_metrics = compute_network_metrics(RG)
reply_summary.to_csv(TABLES / "09_reply_network_summary.csv", index=False)
reply_metrics.to_csv(PROCESSED / "reply_node_metrics.csv", index=False)
nx.write_gexf(RG, NETWORKS / "reply_network.gexf")
display(reply_summary)
display(reply_metrics.head(20))


In [ ]:
progress.step("if MG.number_of_nodes() > 0:")
# In-degree vs out-degree scatter for mention network
if MG.number_of_nodes() > 0:
    in_deg = dict(MG.in_degree(weight="weight"))
    out_deg = dict(MG.out_degree(weight="weight"))
    degree_plot = pd.DataFrame({"node": list(MG.nodes())})
    degree_plot["weighted_in_degree"] = degree_plot["node"].map(in_deg).fillna(0)
    degree_plot["weighted_out_degree"] = degree_plot["node"].map(out_deg).fillna(0)
    degree_plot = degree_plot.merge(mention_metrics[["node", "pagerank", "betweenness_centrality"]], on="node", how="left")
    fig = px.scatter(degree_plot, x="weighted_out_degree", y="weighted_in_degree", size="pagerank", hover_name="node", title="Mention network: attention received vs mentioning activity")
    fig.update_layout(height=620, xaxis_title="Weighted out-degree: mentions sent", yaxis_title="Weighted in-degree: mentions received")
    save_plotly(fig, INTERACTIVE / "09_mention_indegree_outdegree_scatter.html", FIGURES / "09_mention_indegree_outdegree_scatter.png")
    fig.show()


In [ ]:
progress.step("save_pyvis_network(MG, INTERACTIVE / '09_user_mention_network_interactive.html', title='User mention network', node_metr")
save_pyvis_network(MG, INTERACTIVE / "09_user_mention_network_interactive.html", title="User mention network", node_metrics=mention_metrics, max_nodes=900)
save_pyvis_network(RG, INTERACTIVE / "09_reply_network_interactive.html", title="Reply network", node_metrics=reply_metrics, max_nodes=900)
print("Saved interactive user-level networks.")


## Run complete

The final cell closes the progress bar and confirms that all tracked notebook stages finished.


In [ ]:
progress.done("Notebook run completed")
